# 15.12 Recursion and Backtracking

**Prerequisites:** 15.5 Stacks and Queues, 15.7 Trees, 04 Functions  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Base case, recursive case, and the **leap of faith** that makes it writable
- **Euclid's GCD** - recursion at its most elegant
- The call stack, the depth limit, and converting recursion to iteration
- 🔴 When recursion is the wrong tool
- **Memoisation** - and the bridge to dynamic programming (**15.13**)
- **Backtracking**: choose → explore → **un**choose
- Permutations, subsets, combinations
- **N-Queens** with pruning, and what pruning is worth
- Interview questions, worked

---

## The two rules

Every recursive function needs exactly two things:

```
    def solve(problem):
        if <trivially small>:            1. BASE CASE - stop here
            return <the obvious answer>

        return combine(solve(smaller))   2. RECURSIVE CASE - shrink and recurse
```

🔴 **The recursive case must make the problem strictly smaller**, and must eventually reach the base case. Miss either and you get infinite recursion — which in Python is a `RecursionError` rather than a hung machine, and that is a mercy.

### The leap of faith

The hardest part is psychological. Do **not** try to trace the whole call tree in your head — it does not fit.

> **Assume the recursive call already works.** Ask only: *if `solve(smaller)` returns the right answer, how do I build my answer from it?*

That is the entire technique. You verify the base case, you verify one combining step, and induction does the rest.

**Where recursion is natural:** anything defined in terms of itself. Trees (**15.7**) — a tree *is* a node plus two smaller trees. Graphs (**15.9**). Divide and conquer (**15.10**). Nested structures like JSON (**8.3**).

## Euclid's algorithm - the classic

The greatest common divisor of two numbers, from around 300 BC, and still the fastest known method.

**The insight:**

> Any number dividing both `a` and `b` also divides `a - b`, and therefore `a mod b`.
>
> So `gcd(a, b) == gcd(b, a mod b)` — and the numbers shrink fast.

```
   gcd(48, 18)
     = gcd(18, 48 mod 18) = gcd(18, 12)
     = gcd(12, 18 mod 12) = gcd(12, 6)
     = gcd(6,  12 mod 6)  = gcd(6, 0)
     = 6                              base case: gcd(x, 0) == x
```

Three lines, and **O(log(min(a, b)))** — the numbers shrink at least as fast as the Fibonacci sequence grows.

> The worst case is two **consecutive Fibonacci numbers**, which is a pleasing enough fact to be worth knowing: `gcd(fib(n), fib(n-1))` takes the maximum number of steps for numbers of that size.

In [ ]:
def gcd_recursive(a, b, depth=0):
    """Euclid. gcd(a, 0) = a; otherwise gcd(b, a mod b)."""
    if b == 0:
        return a, depth                     # base case
    return gcd_recursive(b, a % b, depth + 1)


def gcd_iterative(a, b):
    """The same algorithm as a loop. O(1) stack space."""
    steps = 0
    while b:
        a, b = b, a % b
        steps += 1
    return a, steps


def lcm(a, b):
    """Lowest common multiple, from gcd - a x b = gcd x lcm."""
    return a * b // gcd_iterative(a, b)[0]


import math

print(f"{'a':>10}{'b':>10}{'gcd':>8}{'steps':>8}{'math.gcd':>10}")
print("-" * 46)
for a, b in ((48, 18), (17, 5), (270, 192), (1_000_000, 999_999), (12, 0)):
    value, steps = gcd_iterative(a, b)
    print(f"{a:>10,}{b:>10,}{value:>8}{steps:>8}{math.gcd(a, b):>10}")

print("\nrecursive and iterative agree:",
      all(gcd_recursive(a, b)[0] == gcd_iterative(a, b)[0]
          for a in range(1, 60) for b in range(0, 60)))

print("\nlcm(4, 6) =", lcm(4, 6), "| lcm(21, 6) =", lcm(21, 6))

# the worst case: consecutive Fibonacci numbers
fibs = [1, 1]
while len(fibs) < 30:
    fibs.append(fibs[-1] + fibs[-2])

print("\nthe worst case for Euclid is consecutive Fibonacci numbers:")
for i in (10, 20, 29):
    _, steps = gcd_iterative(fibs[i], fibs[i - 1])
    print(f"  gcd({fibs[i]:>7,}, {fibs[i - 1]:>7,}) took {steps:>2} steps")
print("\n  Even then it is logarithmic - roughly 5 steps per extra digit.")

## Recursion is the call stack

From **15.5**: each call pushes a frame holding its locals and where to return. That is why:

| | |
|---|---|
| Recursion costs **O(depth) memory** | frames stack up (**15.1**) |
| Deep recursion raises `RecursionError` | the limit is ~1000 by default |
| A traceback reads bottom-up | it *is* the stack, printed |

🔴 **Python has no tail-call optimisation**, and this is deliberate — Guido has argued that losing tracebacks is not worth the saving. So a tail-recursive function gets **no** benefit over any other; converting to a loop is on you.

### Raising the limit is not the fix

`sys.setrecursionlimit(100000)` is available and is usually the wrong answer: the limit exists to turn a **C-level stack overflow** — which would crash the interpreter — into a catchable Python exception. Raise it too far and you get a hard segfault instead.

**The real fix is an explicit stack** (**15.5**) or a loop.

In [ ]:
import sys

print("recursion limit:", sys.getrecursionlimit())


def countdown_recursive(n):
    if n == 0:
        return 0
    return 1 + countdown_recursive(n - 1)


def countdown_iterative(n):
    total = 0
    while n:
        total += 1
        n -= 1
    return total


for n in (500, 5_000):
    try:
        print(f"  recursive({n:>6,}): {countdown_recursive(n):>6,}")
    except RecursionError:
        print(f"  recursive({n:>6,}): RecursionError")
print(f"  iterative(1,000,000): {countdown_iterative(1_000_000):>6,}")

print("\n🔴 Python does NOT optimise tail calls. This function returns the")
print("   recursive result directly - a perfect tail call - and still")
print("   consumes a stack frame per level.")

# converting recursion to iteration with an explicit stack (15.5)


def sum_nested_recursive(item):
    if isinstance(item, int):
        return item
    return sum(sum_nested_recursive(part) for part in item)


def sum_nested_iterative(item):
    """The same traversal, with the stack moved onto the heap."""
    total = 0
    stack = [item]
    while stack:
        current = stack.pop()
        if isinstance(current, int):
            total += current
        else:
            stack.extend(current)
    return total


nested = [1, [2, [3, [4, [5]]]], 6]
print(f"\n  nested sum, recursive: {sum_nested_recursive(nested)}")
print(f"  nested sum, iterative: {sum_nested_iterative(nested)}")

# a structure too deep for recursion
deep = current = []
for _ in range(5_000):
    nxt = []
    current.append(1)
    current.append(nxt)
    current = nxt

try:
    sum_nested_recursive(deep)
    print("  5,000 deep, recursive: fine")
except RecursionError:
    print("  5,000 deep, recursive: RecursionError")
print(f"  5,000 deep, iterative: {sum_nested_iterative(deep):,}")

## 🔴 Memoisation - and the bridge to 15.13

**15.1** showed naive `fib(30)` taking 2,692,537 calls, and `@functools.cache` reducing it to 31. Here is *why*.

```
                fib(5)
              /        \
         fib(4)         fib(3)      <- fib(3) computed TWICE
        /      \        /     \
   fib(3)    fib(2)  fib(2)  fib(1) <- fib(2) computed THREE times
```

The call tree has **overlapping subproblems** — the same question asked repeatedly. Memoisation caches each answer the first time.

| | Calls | Complexity |
|---|---|---|
| Naive | ~2ⁿ | O(2ⁿ) |
| Memoised | n | **O(n)** |

> **Overlapping subproblems is exactly the precondition for dynamic programming.** Memoisation *is* top-down DP — **15.13** takes it further and turns it bottom-up.

🔴 **`functools.cache` requires hashable arguments** (**15.6**) — a list argument raises `TypeError`. Convert to a tuple, or cache on a key you construct.

In [ ]:
import functools

calls = {"naive": 0, "memo": 0}


def fib_naive(n):
    calls["naive"] += 1
    if n < 2:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


@functools.cache                          # 3.9+; lru_cache(maxsize=None) before
def fib_memo(n):
    calls["memo"] += 1
    if n < 2:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)


print(f"{'n':>5}{'naive calls':>14}{'memo calls':>13}")
print("-" * 32)
for n in (10, 20, 25, 30):
    calls["naive"] = 0
    fib_naive(n)
    before = calls["memo"]
    fib_memo(n)
    print(f"{n:>5}{calls['naive']:>14,}{calls['memo'] - before:>13,}")

print("\n  The memoised column shrinks to zero as the cache fills - later")
print("  values reuse everything computed for earlier ones.")
print("  cache info:", fib_memo.cache_info())

print(f"\n  fib(300) memoised = {str(fib_memo(300))[:30]}... "
      f"({len(str(fib_memo(300)))} digits)")
print("  The naive version would take longer than the age of the universe.")

# 🔴 the hashability requirement


@functools.cache
def total(values):
    return sum(values)


print("\n  with a tuple :", total((1, 2, 3)))
try:
    total([1, 2, 3])
except TypeError as exc:
    print("  with a list  : TypeError:", exc)
    print("  🔴 cache keys must be hashable (15.6). Pass a tuple.")

---

# Backtracking

Systematic trial and error: build a candidate step by step, and the moment it cannot work, **undo the last step and try something else**.

```
    def backtrack(state):
        if <complete>:
            record(state)
            return

        for choice in <options>:
            if not <valid>:  continue     <- PRUNE: skip doomed branches
            make(choice)                  <- CHOOSE
            backtrack(state)              <- EXPLORE
            undo(choice)                  <- UNCHOOSE  🔴 the step people forget
```

🔴 **The undo is the whole algorithm.** Without it, state leaks between branches and every subsequent answer is wrong. It is the single most common backtracking bug.

Backtracking is **DFS over a tree of partial solutions** (**15.9**) — the tree is never built; it exists only as the call stack.

| Use it for | Examples |
|---|---|
| Generating all of something | permutations, subsets, combinations |
| Constraint satisfaction | N-Queens, Sudoku, graph colouring |
| Path finding with constraints | word search, maze solving |

> **Backtracking is exponential.** That is expected — it explores a combinatorial space. **Pruning** is what makes it tractable, and its value is measured below.

In [ ]:
def permutations(items):
    """All orderings. O(n! * n) - there are n! of them."""
    results = []
    current = []
    used = [False] * len(items)

    def backtrack():
        if len(current) == len(items):
            results.append(list(current))       # 🔴 COPY - current keeps changing
            return
        for i, item in enumerate(items):
            if used[i]:
                continue
            used[i] = True
            current.append(item)                # CHOOSE
            backtrack()                         # EXPLORE
            current.pop()                       # UNCHOOSE
            used[i] = False                     # UNCHOOSE

    backtrack()
    return results


def subsets(items):
    """The power set: 2^n subsets. Each item is either in or out."""
    results = []
    current = []

    def backtrack(start):
        results.append(list(current))           # every prefix IS a subset
        for i in range(start, len(items)):
            current.append(items[i])            # CHOOSE
            backtrack(i + 1)                    # EXPLORE - only forward
            current.pop()                       # UNCHOOSE

    backtrack(0)
    return results


def combinations(items, k):
    """All k-sized selections. Prunes branches that cannot reach k."""
    results = []
    current = []

    def backtrack(start):
        if len(current) == k:
            results.append(list(current))
            return
        # PRUNE: stop if too few items remain to ever reach k
        for i in range(start, len(items) - (k - len(current)) + 1):
            current.append(items[i])
            backtrack(i + 1)
            current.pop()

    backtrack(0)
    return results


import itertools

items = ["a", "b", "c"]
print("permutations:", permutations(items))
print("  count:", len(permutations(items)), "= 3! =", math.factorial(3))
print("  matches itertools:",
      permutations(items) == [list(p) for p in itertools.permutations(items)])

print("\nsubsets     :", subsets(items))
print("  count:", len(subsets(items)), "= 2^3 =", 2 ** 3)

print("\ncombinations of 2:", combinations(items, 2))
print("  matches itertools:",
      combinations(items, 2) == [list(c) for c in itertools.combinations(items, 2)])

print("\n🔴 Note `results.append(list(current))`. Appending `current` itself")
print("   would store a REFERENCE to a list that keeps mutating - every")
print("   result would end up identical, and empty.")

broken = []
shared = []
for x in items:
    shared.append(x)
    broken.append(shared)               # the same list, three times
    shared.pop()
print("   demonstrated:", broken, "<- all three are the same object")

## N-Queens, and what pruning is worth

Place n queens on an n×n board so that none attacks another — no shared row, column or diagonal.

```
   n = 4, one solution:      . Q . .
                             . . . Q
                             Q . . .
                             . . Q .
```

**Brute force** would try every placement: C(64, 8) ≈ 4.4 **billion** boards for n=8.

**Backtracking with pruning** places one queen per row and abandons a branch the moment it conflicts — so entire subtrees are never explored.

### The diagonal trick

```
    same ╲ diagonal:  row - col  is constant
    same ╱ diagonal:  row + col  is constant
```

So three sets — columns, `row-col`, `row+col` — give **O(1)** conflict checks instead of scanning the board (**15.6**).

The cell below counts nodes explored with and without pruning.

In [ ]:
def solve_n_queens(n):
    """Returns (solutions, nodes explored). Compare with the unpruned version below."""
    solutions = []
    columns, diagonal, anti_diagonal = set(), set(), set()
    placement = []
    nodes = 0

    def conflicts(row, col):
        return (col in columns
                or (row - col) in diagonal
                or (row + col) in anti_diagonal)

    def backtrack(row):
        nonlocal nodes
        nodes += 1
        if row == n:
            solutions.append(list(placement))
            return
        for col in range(n):
            if conflicts(row, col):
                continue                       # PRUNE the whole subtree
            columns.add(col)                   # CHOOSE
            diagonal.add(row - col)
            anti_diagonal.add(row + col)
            placement.append(col)
            backtrack(row + 1)                 # EXPLORE
            placement.pop()                    # UNCHOOSE
            anti_diagonal.discard(row + col)
            diagonal.discard(row - col)
            columns.discard(col)

    backtrack(0)
    return solutions, nodes


print(f"{'n':>4}{'solutions':>12}{'nodes explored':>17}{'n^n boards':>16}")
print("-" * 50)
for n in range(4, 10):
    solutions, nodes = solve_n_queens(n)
    print(f"{n:>4}{len(solutions):>12,}{nodes:>17,}{n ** n:>16,}")

print("\n  The last column is how many placements exist with one queen per")
print("  row. Pruning explores a tiny fraction of it.")

solutions, _ = solve_n_queens(4)
print(f"\n  the {len(solutions)} solutions for n=4:")
for solution in solutions:
    for row, col in enumerate(solution):
        print("    " + " ".join("Q" if c == col else "." for c in range(4)))
    print()

In [ ]:
# What is pruning actually worth? Compare against checking only at the end.
def n_queens_no_pruning(n):
    """Generate every one-queen-per-row placement, validate at the end."""
    solutions = []
    nodes = 0
    placement = []

    def valid(cols):
        for r1 in range(len(cols)):
            for r2 in range(r1 + 1, len(cols)):
                if cols[r1] == cols[r2] or abs(cols[r1] - cols[r2]) == abs(r1 - r2):
                    return False
        return True

    def generate(row):
        nonlocal nodes
        nodes += 1
        if row == n:
            if valid(placement):
                solutions.append(list(placement))
            return
        for col in range(n):
            placement.append(col)
            generate(row + 1)
            placement.pop()

    generate(0)
    return solutions, nodes


print(f"{'n':>4}{'pruned nodes':>15}{'unpruned nodes':>17}{'saving':>12}")
print("-" * 48)
for n in range(4, 9):
    pruned_solutions, pruned_nodes = solve_n_queens(n)
    plain_solutions, plain_nodes = n_queens_no_pruning(n)
    assert len(pruned_solutions) == len(plain_solutions)      # same answers
    print(f"{n:>4}{pruned_nodes:>15,}{plain_nodes:>17,}"
          f"{plain_nodes / pruned_nodes:>11.0f}x")

print("\n  Identical answers, and the gap widens fast with n.")
print("\n🔴 Both are exponential - pruning does not change the complexity")
print("   class. It changes the CONSTANT enough to make n=8 instant")
print("   instead of unbearable, which is the difference that matters.")

## Recursion or iteration?

| Prefer **recursion** when | Prefer **iteration** when |
|---|---|
| The data is recursive — trees, nested JSON, graphs | The problem is a simple sequence |
| The recursive form is dramatically clearer | Depth could exceed ~1000 |
| Backtracking — the stack *is* your state | Performance is critical (call overhead) |
| Divide and conquer | You can express it as a loop without contortion |

### The honest position

> **Write whichever is clearer, then convert if you must.** A recursive tree traversal is five lines and obviously correct; the iterative one needs an explicit stack and is easier to get wrong.

Convert to iteration when the depth is unbounded, or when profiling says the call overhead matters. Do not convert pre-emptively — you will trade clarity for a saving you have not measured (**15.1**).

🔴 **Every recursion can be made iterative** with an explicit stack, because that is exactly what the call stack is (**15.5**). Some conversions are trivial (tail calls); some are genuinely unpleasant (tree post-order).

## Interview questions

**1. What are the two parts of a recursive function?**
> A base case and a recursive case that strictly shrinks the problem. Say what happens without each.

**2. Implement GCD.** *(above)*
> Euclid, recursively or iteratively. O(log min(a,b)). Mention `math.gcd`.

**3. Why does deep recursion fail in Python, and what would you do?**
> Each call costs a stack frame; the limit is ~1000 to convert a C stack overflow into a catchable exception. Rewrite with an explicit stack. Note Python has no TCO.

**4. Generate all permutations / subsets / combinations.** *(above)*
> Backtracking with choose/explore/unchoose. Copy the state when recording, and know the counts: n!, 2ⁿ, C(n,k).

**5. Solve N-Queens.** *(above)*
> One queen per row, three sets for O(1) conflict checks, prune immediately.

**6. What is memoisation, and when does it help?**
> Caching results by argument. It helps when subproblems **overlap** — which is the precondition for DP (**15.13**). It does nothing for problems with no repeats, like merge sort.

**7. Word search in a grid.**
> Backtracking from each cell, marking visited and **unmarking on the way out**. The unmark is the bug people ship.

**8. Generate all valid parentheses combinations of n pairs.**
> Backtracking with two counters; prune when closing would exceed opening. The count is the nth Catalan number.

**9. Flatten a deeply nested list.**
> Recursive is clearest; iterative with a stack if depth is unbounded.

**10. What is the complexity of backtracking?**
> Exponential in general — O(n!) for permutations, O(2ⁿ) for subsets. Pruning reduces the constant, not the class.

In [ ]:
# Questions 7 and 8 - both classic, both short.
def word_search(board, word):
    """Backtracking over a grid. The unmark on the way out is essential."""
    if not board or not word:
        return False
    rows, cols = len(board), len(board[0])

    def explore(row, col, index):
        if index == len(word):
            return True
        if not (0 <= row < rows and 0 <= col < cols):
            return False
        if board[row][col] != word[index]:
            return False

        original = board[row][col]
        board[row][col] = "#"                  # CHOOSE (mark visited)
        found = any(explore(row + dr, col + dc, index + 1)
                    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)))
        board[row][col] = original             # 🔴 UNCHOOSE - restore it
        return found

    return any(explore(r, c, 0) for r in range(rows) for c in range(cols))


grid = [list("ABCE"), list("SFCS"), list("ADEE")]
for word in ("ABCCED", "SEE", "ABCB", "ASA"):
    print(f"  search {word!r:<10} -> {word_search(grid, word)}")
print("  🔴 'ABCB' is False: the B cannot be reused. That is what the")
print("     visited mark enforces - and the restore is what lets a")
print("     DIFFERENT branch use that cell later.")
print("  grid unchanged after searching:",
      grid == [list("ABCE"), list("SFCS"), list("ADEE")])


def generate_parentheses(n):
    """All valid combinations of n pairs. Prune invalid prefixes early."""
    results = []
    current = []

    def backtrack(opened, closed):
        if len(current) == 2 * n:
            results.append("".join(current))
            return
        if opened < n:                         # can always open another
            current.append("(")
            backtrack(opened + 1, closed)
            current.pop()
        if closed < opened:                    # PRUNE: never close too many
            current.append(")")
            backtrack(opened, closed + 1)
            current.pop()

    backtrack(0, 0)
    return results


print()
for n in (1, 2, 3):
    combos = generate_parentheses(n)
    print(f"  n={n}: {combos}")

catalan = [1, 1, 2, 5, 14, 42]
print("\n  counts:", [len(generate_parentheses(n)) for n in range(1, 6)])
print("  Catalan:", catalan[1:6])
print("\n  The `closed < opened` test prunes every prefix that can never")
print("  become valid - which is why only valid strings are ever built.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Forgetting to undo the choice.** State leaks into sibling branches and every later answer is wrong. This is the backtracking bug.
2. 🔴 **Appending the mutable state instead of a copy.** All results become references to one list that ends up empty.
3. 🔴 **No base case, or one that is unreachable.** `RecursionError`.
4. 🔴 **Raising the recursion limit to 'fix' depth.** The limit protects you from a C stack overflow; raise it far enough and you segfault. Use an explicit stack.
5. **Expecting tail-call optimisation.** Python has none, deliberately.
6. **Memoising a function with unhashable arguments.** `functools.cache` needs hashable keys - pass tuples (**15.6**).
7. **Memoising a function with no overlapping subproblems.** You pay for the cache and gain nothing.
8. **Memoising an impure function.** Caching a function that reads changing state returns stale answers.
9. **Recursing on a data structure that might contain a cycle.** Infinite recursion - carry a visited set (**15.9**).

## Best Practices

- Write the base case first, then the recursive case.
- Trust the recursive call - do not trace the whole tree in your head.
- Follow choose / explore / unchoose literally, and put the unchoose immediately after the recursive call.
- Copy mutable state when recording a result.
- Prune as early as possible - it does not change the complexity class, and it changes the runtime enormously.
- Use `functools.cache` for memoisation rather than a hand-rolled dict.
- Convert to iteration when depth is unbounded, not before.
- Test the empty input, a single element, and the maximum depth you expect.

## Practice Exercises

Try these before moving on.

1. Implement the extended Euclidean algorithm, returning `x` and `y` such that `ax + by = gcd(a, b)`.
2. 🔴 Remove `current.pop()` from `permutations` and predict the output before running it. Then explain exactly what went wrong.
3. Implement `combinations_with_replacement` by changing one argument in the recursive call. Which one?
4. Solve Sudoku with backtracking. Which pruning gives the biggest win - and can you measure it as N-Queens did above?
5. Implement 'letter combinations of a phone number'. What is the complexity in terms of the number of digits?
6. Write `permutations` iteratively with an explicit stack. Is it clearer? Would you ship it?
7. 🔴 Memoise a function that takes a list argument, by converting to a tuple inside a wrapper. Why can you not simply decorate the original?
8. Compare `solve_n_queens(12)` with and without pruning - but predict the unpruned node count first, and put a time limit on it before you run it.